In [1]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# import wandb
# from kaggle_secrets import UserSecretsClient

In [2]:
# try:
#     secrets = UserSecretsClient()
#     wandb_key = secrets.get_secret("WANDB_API_KEY")
#     wandb.login(key=wandb_key)
#     print("W&B login successful!")
# except Exception as e:
#     wandb.login(anonymous="allow")

In [3]:
DATA_DIR = "/kaggle/input/datasets/sreekaranreddy2005/mel-spectograms/mel_spectrograms"
CSV_PATH = "/kaggle/input/datasets/sreekaranreddy2005/mel-spectograms-labels/mel_label.csv"
OUTPUT_DIR = "/kaggle/working"
WANDB_PROJECT = "24f2000010-t12026"  

# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 10
SEED = 42
N_FOLDS = 5
TRAIN_FOLD = 0

In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [5]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

In [6]:
# print("Loading data")
# df = pd.read_csv(CSV_PATH)

# if 'image' in df.columns and 'labels' in df.columns:
#     df = df[['image', 'labels']].copy()
# elif len(df.columns) == 3:
#     df.columns = ['image', 'empty', 'labels']
#     df = df[['image', 'labels']].copy()

# df['song_group'] = df['image'].apply(lambda x: '_'.join(x.replace('.png', '').split('_')[:2]))
# df['label_idx'] = df['labels'].map(GENRE_TO_IDX)

# print(f"Total samples: {len(df)}")
# print(f"Unique songs: {df['song_group'].nunique()}")
# print(f"Genre distribution:\n{df['labels'].value_counts()}")

In [7]:
# sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
# for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(df, df['label_idx'], groups=df['song_group'])):
#     if fold_idx == TRAIN_FOLD:
#         train_df = df.iloc[train_idx].reset_index(drop=True)
#         val_df = df.iloc[val_idx].reset_index(drop=True)
#         break

# print(f"\nFold {TRAIN_FOLD}: Train={len(train_df)}, Val={len(val_df)}")
# print(f"Train genres: {dict(Counter(train_df['labels']))}")
# print(f"Val genres:   {dict(Counter(val_df['labels']))}")


In [8]:
# class SpecAugment:
#     """Apply SpecAugment-like masking to mel spectrogram images (as tensors)."""
#     def __init__(self, freq_masks=2, time_masks=2, freq_width=20, time_width=30):
#         self.freq_masks = freq_masks
#         self.time_masks = time_masks
#         self.freq_width = freq_width
#         self.time_width = time_width

#     def __call__(self, img_tensor):
#         # img_tensor shape: (C, H, W)
#         _, h, w = img_tensor.shape
#         for _ in range(self.freq_masks):
#             f = random.randint(1, self.freq_width)
#             f0 = random.randint(0, max(0, h - f))
#             img_tensor[:, f0:f0+f, :] = 0
#         for _ in range(self.time_masks):
#             t = random.randint(1, self.time_width)
#             t0 = random.randint(0, max(0, w - t))
#             img_tensor[:, :, t0:t0+t] = 0
#         return img_tensor

In [9]:
# class MelSpectrogramDataset(Dataset):
#     def __init__(self, dataframe, img_dir, transform=None, spec_augment=None):
#         self.df = dataframe
#         self.img_dir = img_dir
#         self.transform = transform
#         self.spec_augment = spec_augment

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         img_path = os.path.join(self.img_dir, row['image'])
#         img = Image.open(img_path).convert('RGB')

#         if self.transform:
#             img = self.transform(img)

#         if self.spec_augment is not None:
#             img = self.spec_augment(img)

#         label = row['label_idx']
#         return img, label

In [10]:
# train_transform = T.Compose([
#     T.Resize((IMG_SIZE, IMG_SIZE)),
#     T.RandomHorizontalFlip(p=0.5),
#     T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
#     T.ToTensor(),
#     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
# ])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [11]:
# spec_aug = SpecAugment(freq_masks=2, time_masks=2, freq_width=15, time_width=25)

# train_dataset = MelSpectrogramDataset(train_df, DATA_DIR, train_transform, spec_augment=spec_aug)
# val_dataset = MelSpectrogramDataset(val_df, DATA_DIR, val_transform, spec_augment=None)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
#                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
#                         num_workers=NUM_WORKERS, pin_memory=True)

# print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

def build_model():
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model

model = build_model().to(DEVICE)

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 186MB/s]


In [12]:
# wandb.init(
#     project=WANDB_PROJECT,
#     name="model1-cnn-efficientnet-b2",
#     config={
#         "model": "EfficientNet-B2",
#         "img_size": IMG_SIZE,
#         "batch_size": BATCH_SIZE,
#         "epochs": EPOCHS,
#         "lr": LR,
#         "weight_decay": WEIGHT_DECAY,
#         "num_classes": NUM_CLASSES,
#         "seed": SEED,
#         "fold": TRAIN_FOLD,
#         "n_folds": N_FOLDS,
#         "optimizer": "AdamW",
#         "scheduler": "CosineAnnealingWarmRestarts",
#         "augmentations": "SpecAugment+Mixup+ColorJitter+HFlip",
#         "label_smoothing": 0.1,
#         "train_samples": len(train_df),
#         "val_samples": len(val_df),
#         "total_params": sum(p.numel() for p in model.parameters()),
#     }
# )

In [13]:
# def mixup_data(x, y, alpha=0.4):
#     if alpha > 0:
#         lam = np.random.beta(alpha, alpha)
#     else:
#         lam = 1.0
#     batch_size = x.size(0)
#     index = torch.randperm(batch_size).to(x.device)
#     mixed_x = lam * x + (1 - lam) * x[index]
#     y_a, y_b = y, y[index]
#     return mixed_x, y_a, y_b, lam

# def mixup_criterion(criterion, pred, y_a, y_b, lam):
#     return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [14]:
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
# optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# best_f1 = 0.0
# best_model_path = os.path.join(OUTPUT_DIR, "model1_cnn_best.pth")
# train_losses, val_losses = [], []
# train_f1s, val_f1s = [], []


# for epoch in range(EPOCHS):
#     model.train()
#     running_loss = 0.0
#     all_preds, all_labels = [], []

#     for batch_idx, (images, labels) in enumerate(train_loader):
#         images, labels = images.to(DEVICE), labels.to(DEVICE)

#         # Apply mixup with 50% probability
#         use_mixup = random.random() < 0.5
#         if use_mixup:
#             images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.4)
#             outputs = model(images)
#             loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
#         else:
#             outputs = model(images)
#             loss = criterion(outputs, labels)

#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()

#         running_loss += loss.item()

#         if not use_mixup:
#             preds = outputs.argmax(dim=1).cpu().numpy()
#             all_preds.extend(preds)
#             all_labels.extend(labels.cpu().numpy())

#     scheduler.step()
#     train_loss = running_loss / len(train_loader)
#     train_f1 = f1_score(all_labels, all_preds, average='macro') if all_labels else 0.0
#     train_losses.append(train_loss)
#     train_f1s.append(train_f1)

#     model.eval()
#     val_loss = 0.0
#     all_val_preds, all_val_labels = [], []

#     with torch.no_grad():
#         for images, labels in val_loader:
#             images, labels = images.to(DEVICE), labels.to(DEVICE)
#             outputs = model(images)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()

#             preds = outputs.argmax(dim=1).cpu().numpy()
#             all_val_preds.extend(preds)
#             all_val_labels.extend(labels.cpu().numpy())

#     val_loss /= len(val_loader)
#     val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
#     val_losses.append(val_loss)
#     val_f1s.append(val_f1)

#     lr_now = optimizer.param_groups[0]['lr']
#     print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
#           f"Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | "
#           f"Val Loss: {val_loss:.4f} F1: {val_f1:.4f} | "
#           f"LR: {lr_now:.6f}")

#    
#     wandb.log({
#         "epoch": epoch + 1,
#         "train/loss": train_loss,
#         "train/macro_f1": train_f1,
#         "val/loss": val_loss,
#         "val/macro_f1": val_f1,
#         "lr": lr_now,
#         "best_val_f1": max(best_f1, val_f1),
#     })

#   
#     if val_f1 > best_f1:
#         best_f1 = val_f1
#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'val_f1': val_f1,
#             'val_loss': val_loss,
#         }, best_model_path)

In [15]:
# print(f"  → Saved best model (F1: {val_f1:.4f})")
# print(f"Best Validation Macro F1: {best_f1:.4f}")

# print()
# checkpoint = torch.load(best_model_path)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

# all_val_preds, all_val_labels = [], []
# with torch.no_grad():
#     for images, labels in val_loader:
#         images = images.to(DEVICE)
#         outputs = model(images)
#         preds = outputs.argmax(dim=1).cpu().numpy()
#         all_val_preds.extend(preds)
#         all_val_labels.extend(labels.numpy())

# report_str = classification_report(all_val_labels, all_val_preds, target_names=GENRES)
# print("\nClassification Report:")
# print(report_str)

In [16]:
# wandb.finish()

In [17]:
import librosa
import librosa.display

TEST_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
SAMPLE_SUBMISSION = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv"
TEST_CSV = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
CHUNK_DURATION = 5 
SR = 22050          
N_MELS_INFER = 128
N_FFT_INFER = 2048
HOP_LEN_INFER = 512

In [18]:
checkpoint = torch.load("/kaggle/input/models/sreekaranreddy2005/model1-cnn/pytorch/default/1/model1_cnn_best.pth", map_location='cuda')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [19]:
def audio_to_mel_image(y, sr, n_mels=128, n_fft=2048, hop_length=512, img_size=224):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 0:
        mel_norm = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_norm = np.zeros_like(mel_db)
    mel_img = (mel_norm * 255).astype(np.uint8)

    img = Image.fromarray(mel_img)
    img = img.transpose(Image.FLIP_TOP_BOTTOM)
    img = img.resize((img_size, img_size), Image.LANCZOS).convert('RGB')
    return img

In [20]:
def chunk_audio(y, sr, chunk_duration=3):
    chunk_samples = sr * chunk_duration
    chunks = []
    for start in range(0, len(y), chunk_samples):
        chunk = y[start:start + chunk_samples]
        if len(chunk) >= chunk_samples // 2: 
            if len(chunk) < chunk_samples:
                chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
            chunks.append(chunk)
    return chunks

In [21]:
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.wav')])

In [22]:
results = []

for i, filename in enumerate(test_files):
    filepath = os.path.join(TEST_DIR, filename)

    y, sr = librosa.load(filepath, sr=SR)

    chunks = chunk_audio(y, sr, CHUNK_DURATION)

    chunk_probs = []
    for chunk in chunks:

        img = audio_to_mel_image(chunk, sr, N_MELS_INFER, N_FFT_INFER, HOP_LEN_INFER, IMG_SIZE)

        img_tensor = val_transform(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(img_tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            chunk_probs.append(probs[0])

    avg_probs = np.mean(chunk_probs, axis=0)
    predicted_idx = np.argmax(avg_probs)
    predicted_genre = IDX_TO_GENRE[predicted_idx]
    confidence = avg_probs[predicted_idx]

    song_id = int(os.path.splitext(filename)[0].replace('song', ''))
    
    results.append({
        'id': song_id,
        'genre': predicted_genre,
    })

    if (i + 1) % 100 == 0 or (i + 1) == len(test_files):
        print(f"  Processed {i+1}/{len(test_files)} songs | "
              f"Last: {filename} → {predicted_genre} ({confidence:.3f})")

results_df = pd.DataFrame(results)
# print(f"\nPredictions complete: {len(results_df)} songs")
# print(f"Genre distribution:\n{results_df['genre'].value_counts()}")

  Processed 100/3020 songs | Last: song0100.wav → country (0.474)
  Processed 200/3020 songs | Last: song0200.wav → metal (0.553)
  Processed 300/3020 songs | Last: song0300.wav → reggae (0.638)
  Processed 400/3020 songs | Last: song0400.wav → metal (0.470)
  Processed 500/3020 songs | Last: song0500.wav → pop (0.881)
  Processed 600/3020 songs | Last: song0600.wav → rock (0.754)
  Processed 700/3020 songs | Last: song0700.wav → pop (0.657)
  Processed 800/3020 songs | Last: song0800.wav → pop (0.856)
  Processed 900/3020 songs | Last: song0900.wav → blues (0.562)
  Processed 1000/3020 songs | Last: song1000.wav → hiphop (0.720)
  Processed 1100/3020 songs | Last: song1100.wav → disco (0.650)
  Processed 1200/3020 songs | Last: song1200.wav → country (0.529)
  Processed 1300/3020 songs | Last: song1300.wav → country (0.591)
  Processed 1400/3020 songs | Last: song1400.wav → classical (0.891)
  Processed 1500/3020 songs | Last: song1500.wav → blues (0.876)
  Processed 1600/3020 songs |

In [23]:
results_df.to_csv("submission.csv",index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
